# Самостоятельная работа 02. Метрики качества классификации

Цель: посчитать матрицу ошибок, долю правильных ответов, точность, полноту, F1-меру и ROC AUC своими руками и с scikit-learn, подобрать порог под требование задачи и объяснить, о чём говорит каждое число.

Правила:

- защищать можно любое число заданий — от одного до всех;
- уйти с занятия раньше времени можно только после защиты хотя бы одного задания;
- баллы ставятся только за защищённые задания: базовое — 1 балл, продвинутое — 3 балла; выполненное, но не защищённое задание даёт 0;
- на защите нужно объяснить любую строку своего кода и при преподавателе изменить его;
- ИИ-помощником можно пользоваться точечно — вспомнить функцию, разобраться с ошибкой; код, который вы не можете объяснить, не засчитывается;
- перед защитой перезапустите ядро (Kernel → Restart) и выполните сверху вниз подготовку данных и защищаемые задания — их самопроверки должны пройти.

Материалы:

- лекция 03 «Классификация и метрики качества»: матрица ошибок, точность и полнота, F1-мера, порог принятия решения, ROC-кривая;
- Орельен Жерон, «Прикладное машинное обучение с помощью Scikit-Learn и TensorFlow», гл. 3, стр. 125–139;
- блокнот практического занятия 04 «Классификация и метрики качества».

| № | Что сделать | Тип | Баллы |
|---|---|---|---|
| 1 | Матрица ошибок и ловушка доли правильных ответов | базовое | 1 |
| 2 | Точность, полнота и F1-мера своими руками | базовое | 1 |
| 3 | Порог под требуемую полноту | базовое | 1 |
| 4 | ROC-кривая, ROC AUC и средняя точность | продвинутое | 3 |

In [ ]:
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

sys.version, np.__version__, pd.__version__, sklearn.__version__, matplotlib.__version__

## Данные

Система видеоаналитики проверяет кадры с камер склада и поднимает тревогу, если на кадре дым. Дым редок — около 5 % кадров. Положительный класс — «дым» (метка 1), отрицательный — «чистый кадр» (метка 0).

Модель уже обучена — это линейный классификатор (логистическая регрессия); как она устроена, сейчас не важно. Как на лекции, для каждого кадра есть оценка решающей функции, полученная перекрёстной проверкой, то есть моделью, которая этот кадр при обучении не видела. Ячейка ниже готова — просто выполните её. В ней:

- `y_true` — фактические метки 4000 кадров;
- `smoke_scores` — оценки решающей функции: чем больше, тем увереннее модель видит дым;
- `y_pred` — прогнозы модели при пороге 0: дым, если оценка больше нуля.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict


def make_smoke_frames(n_frames, seed=0):
    """Генерирует признаки кадров видеопотока и разметку «на кадре есть дым».

    Args:
        n_frames: Число кадров.
        seed: Зерно генератора случайных чисел.

    Returns:
        Матрица признаков кадров и вектор меток: 1 — дым есть, 0 — нет.
    """
    features, labels = make_classification(
        n_samples=n_frames,
        n_features=6,
        n_informative=3,
        n_redundant=1,
        n_clusters_per_class=1,
        weights=[0.95],  # Дым редок: около 5 % кадров
        flip_y=0.0,
        class_sep=1.5,
        random_state=seed,
    )
    return features, labels


frame_features, y_true = make_smoke_frames(n_frames=4000, seed=42)
smoke_scores = cross_val_predict(LogisticRegression(), frame_features, y_true, cv=3, method='decision_function')
y_pred = (smoke_scores > 0).astype(int)

len(y_true), y_true.sum(), y_pred.sum()

Вывод: всего кадров, из них с дымом, и сколько раз модель подняла тревогу при пороге 0.

## Задание 1 (базовое, 1 балл)

Детектор дыма ошибается двумя способами: поднимает ложную тревогу на чистом кадре или пропускает кадр с дымом. Матрица ошибок (англ. *confusion matrix*) считает оба вида ошибок отдельно. Посчитайте её сами и сравните модель с «глупым» детектором, который никогда не видит дыма.

**Дано:** `y_true` — фактические метки кадров, `y_pred` — прогнозы модели при пороге 0.

**Получить:**

- функцию `confusion_counts(y_true, y_pred)`, которая возвращает кортеж `(tn, fp, fn, tp)` — четыре клетки матрицы ошибок, положительный класс — 1;
- `y_pred_constant` — прогнозы константного детектора «дыма нет» для всех кадров;
- `counts_constant` и `counts_model` — результаты `confusion_counts` для константного детектора и для модели;
- `accuracy_constant` и `accuracy_model` — доли правильных ответов (англ. *accuracy*) обоих детекторов, посчитанные по четырём клеткам матрицы.

**Как действовать:**

1. Для каждой из четырёх клеток запишите условие на пару «факт — прогноз»: например, ложноположительный результат (англ. *false positive*, FP) — факт 0, прогноз 1. Число кадров, для которых условие выполнено, — сумма булевой маски.
2. Соберите прогнозы константного детектора: массив нулей той же длины, что `y_true`.
3. Посчитайте клетки матрицы для обоих детекторов.
4. Выразите долю правильных ответов через клетки матрицы: какие из них — верные ответы?

`confusion_matrix`, `accuracy_score` и другие функции `sklearn.metrics` здесь не используйте: задание про то, из чего складывается матрица.

**Что почитать:**

- лекция 03, слайды «Константная модель «всегда не 5»», «Почему доля правильных ответов обманывает», «Четыре исхода прогноза» и «Матрица детектора пятёрок»; книга, стр. 126–129;
- [Ошибки первого и второго рода](https://ru.wikipedia.org/wiki/Ошибки_первого_и_второго_рода) — ложноположительный и ложноотрицательный результат в статистике;
- [Confusion matrix](https://en.wikipedia.org/wiki/Confusion_matrix) и [Accuracy paradox](https://en.wikipedia.org/wiki/Accuracy_paradox) — английская Википедия: матрица ошибок и почему доля правильных ответов обманывает;
- [`np.sum`](https://numpy.org/doc/1.24/reference/generated/numpy.sum.html) — сумма булевой маски равна числу `True`; [`np.logical_and`](https://numpy.org/doc/1.24/reference/generated/numpy.logical_and.html) — «и» для двух масок, то же делает оператор `&`; [`np.zeros_like`](https://numpy.org/doc/1.24/reference/generated/numpy.zeros_like.html).

Предсказание: не запуская код, оцените, какую долю правильных ответов получит константный детектор «дыма нет» и сколько кадров с дымом он найдёт.

*Ваш ответ:* …

In [ ]:
def confusion_counts(y_true, y_pred):
    """Считает четыре клетки матрицы ошибок двоичного классификатора.

    Args:
        y_true: Фактические метки, 1 — положительный класс.
        y_pred: Спрогнозированные метки.

    Returns:
        Кортеж (tn, fp, fn, tp).
    """
    ...  # Ваш код


y_pred_constant = ...  # Ваш код
counts_constant = ...  # Ваш код
counts_model = ...  # Ваш код
accuracy_constant, accuracy_model = ..., ...  # Ваш код

counts_constant, counts_model, accuracy_constant, accuracy_model

In [ ]:
toy_true = np.array([1, 1, 1, 0, 0, 0, 0])
toy_pred = np.array([1, 0, 0, 1, 0, 0, 0])
toy_counts = tuple(int(count) for count in confusion_counts(toy_true, toy_pred))
assert toy_counts == (3, 1, 2, 1), f'На примере из 7 кадров нужно TN=3, FP=1, FN=2, TP=1, а получено {toy_counts}'
assert sum(counts_model) == len(y_true), 'Сумма клеток матрицы должна равняться числу кадров'
assert counts_constant[1] == 0 and counts_constant[3] == 0, 'Константный детектор «дыма нет» не даёт ни одной тревоги'
assert np.isclose(accuracy_model, np.mean(y_pred == y_true)), 'Доля правильных ответов модели посчитана неверно'
assert 0.9 < accuracy_constant < accuracy_model < 1, 'Доли правильных ответов — между 0,9 и 1, у модели — выше'

'Задание 1: самопроверка пройдена'

Доля правильных ответов модели лишь немного выше, чем у детектора, который вообще не видит дыма. Объясните по своим матрицам, почему доля правильных ответов не показывает, полезен ли детектор, и какие клетки матрицы это показывают.

*Ваш ответ:* …

## Задание 2 (базовое, 1 балл)

Матрица из четырёх чисел неудобна, когда нужно сравнить детекторы. Точность (англ. *precision*) и полнота (англ. *recall*) сворачивают её в два числа, F1-мера (англ. *F1 score*) — в одно. Напишите их сами и сверьтесь с scikit-learn. В переводе книги accuracy названа «правильностью» — не путайте её с точностью.

**Дано:** `y_true`, `y_pred` — фактические метки и прогнозы модели при пороге 0.

**Получить:**

- функцию `precision_recall_f1(tp, fp, fn)`, которая возвращает кортеж `(precision, recall, f1)`;
- `precision_own`, `recall_own`, `f1_own` — результат вашей функции на клетках матрицы ошибок модели;
- `precision_sklearn`, `recall_sklearn`, `f1_sklearn` — те же метрики, посчитанные функциями `sklearn.metrics` по `y_true` и `y_pred`.

**Как действовать:**

1. Точность — доля верных среди всех тревог детектора; полнота — доля найденных среди всех кадров, где дым действительно есть. Решите, какие клетки матрицы стоят в числителе и знаменателе каждой: на слайде «Четыре исхода прогноза» точность считается по столбцу, полнота — по строке.
2. F1-мера — среднее гармоническое точности и полноты: величина, обратная F1, равна среднему арифметическому величин, обратных точности и полноте.
3. Клетки матрицы модели получите функцией `confusion_matrix` из scikit-learn — так задание не зависит от задания 1. В каком порядке стоят клетки в матрице 2 × 2 и как развернуть её в четыре числа, смотрите в документации.
4. Посчитайте те же три метрики функциями scikit-learn; в документации проверьте, какой аргумент — факт, а какой — прогноз.

Внутри `precision_recall_f1` функции scikit-learn не используйте: задание про формулы.

**Что почитать:**

- лекция 03, слайды «Точность (precision)», «Полнота (recall)», «F1-мера» и «Почему среднее гармоническое»; книга, стр. 128–131;
- [F-мера](https://ru.wikipedia.org/wiki/F-мера) и [Среднее гармоническое](https://ru.wikipedia.org/wiki/Среднее_гармоническое) — Википедия; [Precision and recall](https://en.wikipedia.org/wiki/Precision_and_recall) — английская Википедия;
- [`confusion_matrix`](https://scikit-learn.org/1.3/modules/generated/sklearn.metrics.confusion_matrix.html) — порядок строк и столбцов, пример с `ravel`;
- [`precision_score`](https://scikit-learn.org/1.3/modules/generated/sklearn.metrics.precision_score.html), [`recall_score`](https://scikit-learn.org/1.3/modules/generated/sklearn.metrics.recall_score.html), [`f1_score`](https://scikit-learn.org/1.3/modules/generated/sklearn.metrics.f1_score.html).

In [ ]:
def precision_recall_f1(tp, fp, fn):
    """Считает точность, полноту и F1-меру по клеткам матрицы ошибок.

    Args:
        tp: Число истинно положительных результатов.
        fp: Число ложноположительных результатов.
        fn: Число ложноотрицательных результатов.

    Returns:
        Кортеж (precision, recall, f1).
    """
    ...  # Ваш код


precision_own, recall_own, f1_own = ..., ..., ...  # Ваша функция на клетках confusion_matrix
precision_sklearn, recall_sklearn, f1_sklearn = ..., ..., ...  # Функции sklearn.metrics

precision_own, recall_own, f1_own, precision_sklearn, recall_sklearn, f1_sklearn

In [ ]:
toy_metrics = precision_recall_f1(tp=1, fp=1, fn=2)
assert np.allclose(toy_metrics, (0.5, 1 / 3, 0.4)), f'При TP=1, FP=1, FN=2 нужно (0,5; 0,333; 0,4), а не {toy_metrics}'
own_metrics = [precision_own, recall_own, f1_own]
sklearn_metrics = [precision_sklearn, recall_sklearn, f1_sklearn]
assert np.allclose(own_metrics, sklearn_metrics), 'Ваши метрики не совпали с scikit-learn: проверьте клетки и аргументы'

'Задание 2: самопроверка пройдена'

1. Сколько ложных тревог и сколько пропущенных кадров с дымом даёт модель при пороге 0? Как это видно по точности и полноте?
2. Почему F1-мера модели ближе к меньшей из двух метрик, чем их среднее арифметическое?

*Ваш ответ:* …

## Задание 3 (базовое, 1 балл)

Модель выдаёт не метку, а оценку решающей функции (англ. *decision function*) `smoke_scores`; метка получается сравнением оценки с порогом принятия решения (англ. *decision threshold*), и `y_pred` — это порог 0. Пропущенный пожар обходится гораздо дороже ложной тревоги, поэтому заказчик требует: детектор должен находить не меньше 95 % кадров с дымом. Подберите порог.

**Дано:** `y_true`, `smoke_scores` и сетка порогов `threshold_grid` от −6 до 4 с шагом 0,25 — она уже записана в заготовке.

**Получить:**

- `threshold_table` — таблицу pandas: по строке на каждый порог сетки в порядке возрастания, столбцы `threshold`, `precision`, `recall`;
- `threshold_95` — наибольший порог сетки, при котором полнота не ниже 0,95;
- `precision_95`, `recall_95` — точность и полноту при этом пороге.

**Как действовать:**

1. Сначала запишите предсказание в ячейку ниже.
2. Для каждого порога сетки получите прогнозы «оценка не меньше порога — дым» и посчитайте точность и полноту — функциями scikit-learn или своими формулами из задания 2.
3. Соберите результаты в таблицу.
4. Отберите строки, где полнота не ниже 0,95, и возьмите из них строку с наибольшим порогом.
5. Выполните готовую ячейку с графиком после самопроверки и ответьте на вопросы.

**Что почитать:**

- лекция 03, слайды «Решающая функция и порог», «Порог сдвигает баланс», «Точность и полнота против порога» и «Выбор порога под цель»; книга, стр. 131–135;
- [Чувствительность и специфичность](https://ru.wikipedia.org/wiki/Чувствительность_и_специфичность) — Википедия: полнота в медицине называется чувствительностью;
- [руководство scikit-learn по метрикам](https://scikit-learn.org/1.3/modules/model_evaluation.html) — раздел Precision, recall and F-measures;
- [`pandas.DataFrame`](https://pandas.pydata.org/pandas-docs/version/2.0/reference/api/pandas.DataFrame.html) — таблицу удобно собрать из списка словарей; [отбор строк по условию](https://pandas.pydata.org/pandas-docs/version/2.0/user_guide/indexing.html) — раздел Boolean indexing.

Предсказание: что станет с точностью и полнотой, если поднять порог с 0 до 2? Может ли полнота при этом вырасти?

*Ваш ответ:* …

In [ ]:
threshold_grid = np.arange(-6.0, 4.25, 0.25)

threshold_table = ...  # Ваш код: столбцы threshold, precision, recall
threshold_95, precision_95, recall_95 = ..., ..., ...  # Ваш код

threshold_table, threshold_95, precision_95, recall_95

In [ ]:
smoke_frame_scores = smoke_scores[y_true == 1]
assert len(threshold_table) == len(threshold_grid), 'В таблице должна быть строка на каждый порог сетки'
assert np.all(np.diff(threshold_table['recall']) <= 0), 'С ростом порога полнота не может расти: проверьте сравнение'
assert np.isclose(np.mean(smoke_frame_scores >= threshold_95), recall_95), 'recall_95 не равна полноте при threshold_95'
assert recall_95 >= 0.95, 'При пороге threshold_95 полнота должна быть не ниже 0,95'
assert np.mean(smoke_frame_scores >= threshold_95 + 0.25) < 0.95, 'Есть порог сетки выше, где полнота ещё не ниже 0,95'
row_95 = threshold_table[np.isclose(threshold_table['threshold'], threshold_95)]
assert np.isclose(row_95['precision'].iloc[0], precision_95), 'precision_95 не совпадает с точностью в таблице'

'Задание 3: самопроверка пройдена'

In [ ]:
plt.plot(threshold_table['threshold'], threshold_table['precision'], label='точность')
plt.plot(threshold_table['threshold'], threshold_table['recall'], label='полнота')
plt.axvline(threshold_95, color='gray', linestyle='--', label='порог для полноты 95 %')
plt.xlabel('Порог')
plt.legend()
plt.show()

1. Сколько ложных тревог приходится на один верно найденный кадр с дымом при пороге `threshold_95`? А при пороге 0?
2. По своему графику: почему полнота с ростом порога только убывает, а у точности бывают «выбоины»?

*Ваш ответ:* …

## Задание 4 (продвинутое, 3 балла)

Порог выбирают под задачу, а модели сравнивают по кривым при всех порогах сразу. ROC-кривая (англ. *ROC curve*) откладывает долю истинно положительных TPR — это полнота — против доли ложноположительных FPR, то есть доли чистых кадров, на которых поднята тревога. Площадь под ней, ROC AUC, — одно число: 0,5 у случайного угадывания, 1 у безупречного классификатора. Постройте кривую сами, сверьте площадь с scikit-learn, выясните, что эта площадь означает, и сравните её со средней точностью (англ. *average precision*) — площадью под кривой точности-полноты.

**Дано:** `y_true`, `smoke_scores`.

**Получить:**

- функцию `roc_points(y_true, scores)`, которая возвращает кортеж `(fpr, tpr)` — массивы координат точек ROC-кривой по порогам от самого высокого к самому низкому, первая точка — (0, 0);
- `fpr_own`, `tpr_own` — её результат для модели;
- `auc_own` — площадь под вашей кривой;
- `auc_sklearn` — ROC AUC, посчитанную функцией scikit-learn;
- `auc_pairs` — долю пар «кадр с дымом, чистый кадр», в которых оценка кадра с дымом выше оценки чистого кадра;
- `ap_sklearn` — среднюю точность, посчитанную функцией scikit-learn.

**Как действовать:**

1. Порогами возьмите все различные значения оценок, отсортированные по убыванию. При каждом пороге посчитайте TPR и FPR для прогноза «оценка не меньше порога — дым». Начните кривую с точки (0, 0) — она соответствует порогу выше всех оценок.
2. Посчитайте площадь под кривой методом трапеций.
3. Для `auc_pairs` разделите оценки на две группы — кадры с дымом и чистые кадры — и сравните каждую оценку первой группы с каждой оценкой второй. Трансляция NumPy сравнивает столбец со строкой за одну операцию и даёт матрицу сравнений; циклом по парам не нужно.
4. Посчитайте ROC AUC и среднюю точность функциями scikit-learn — обеим функциям нужны оценки, а не метки.
5. Выполните готовую ячейку с графиками после самопроверки и ответьте на вопросы.

`roc_curve` внутри `roc_points` не используйте: задание про то, как строится кривая.

**Что почитать:**

- лекция 03, слайды «TPR и FPR», «ROC-кривая», «Площадь под ROC-кривой» и «ROC или PR?»; книга, стр. 135–138;
- [ROC-кривая](https://ru.wikipedia.org/wiki/ROC-кривая) и [Метод трапеций](https://ru.wikipedia.org/wiki/Метод_трапеций) — Википедия;
- [`np.unique`](https://numpy.org/doc/1.24/reference/generated/numpy.unique.html), [`np.trapz`](https://numpy.org/doc/1.24/reference/generated/numpy.trapz.html), [трансляция (broadcasting)](https://numpy.org/doc/1.24/user/basics.broadcasting.html);
- [`roc_auc_score`](https://scikit-learn.org/1.3/modules/generated/sklearn.metrics.roc_auc_score.html), [`average_precision_score`](https://scikit-learn.org/1.3/modules/generated/sklearn.metrics.average_precision_score.html); [`roc_curve`](https://scikit-learn.org/1.3/modules/generated/sklearn.metrics.roc_curve.html) — чтобы сверить свою кривую.

In [ ]:
def roc_points(y_true, scores):
    """Считает точки ROC-кривой по всем порогам от самого высокого к самому низкому.

    Args:
        y_true: Фактические метки, 1 — положительный класс.
        scores: Оценки решающей функции.

    Returns:
        Кортеж (fpr, tpr): массивы координат точек кривой, первая точка — (0, 0).
    """
    ...  # Ваш код


fpr_own, tpr_own = roc_points(y_true, smoke_scores)
auc_own = ...  # Ваш код: площадь под вашей кривой
auc_sklearn = ...  # Ваш код: ROC AUC из scikit-learn
auc_pairs = ...  # Ваш код: доля пар, где кадр с дымом оценён выше чистого
ap_sklearn = ...  # Ваш код: средняя точность из scikit-learn

auc_own, auc_sklearn, auc_pairs, ap_sklearn

In [ ]:
assert fpr_own[0] == 0 and tpr_own[0] == 0, 'Кривая должна начинаться в точке (0, 0)'
assert np.isclose(fpr_own[-1], 1) and np.isclose(tpr_own[-1], 1), 'Кривая должна заканчиваться в точке (1, 1)'
assert np.all(np.diff(fpr_own) >= 0) and np.all(np.diff(tpr_own) >= 0), 'При снижении порога FPR и TPR не убывают'
assert np.isclose(auc_own, auc_sklearn, atol=1e-6), 'Площадь под вашей кривой не совпала с ROC AUC scikit-learn'
assert np.isclose(auc_pairs, auc_sklearn, atol=1e-6), 'Доля пар не совпала с ROC AUC: проверьте, что с чем сравниваете'
assert 0 < ap_sklearn < 1, 'Средняя точность — число от 0 до 1; ей нужны оценки, а не метки'

'Задание 4: самопроверка пройдена'

In [ ]:
from sklearn.metrics import precision_recall_curve

precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_true, smoke_scores)
figure, (roc_axis, pr_axis) = plt.subplots(1, 2, figsize=(11, 4))
roc_axis.plot(fpr_own, tpr_own, label=f'модель, ROC AUC = {auc_own:.3f}')
roc_axis.plot([0, 1], [0, 1], 'k--', label='случайное угадывание')
roc_axis.set_xlabel('FPR — доля чистых кадров с тревогой')
roc_axis.set_ylabel('TPR — полнота')
roc_axis.legend()
pr_axis.plot(recall_curve, precision_curve, label=f'модель, AP = {ap_sklearn:.3f}')
pr_axis.axhline(y_true.mean(), color='k', linestyle='--', label='случайное угадывание')
pr_axis.set_xlabel('Полнота')
pr_axis.set_ylabel('Точность')
pr_axis.legend()
plt.show()

1. Что означает число `auc_pairs` и почему оно совпало с площадью под ROC-кривой?
2. ROC AUC модели выше 0,95, и детектор кажется почти идеальным. Почему средняя точность заметно ниже и какая из двух кривых честнее показывает качество детектора дыма?

*Ваш ответ:* …

## Готовность к защите

Перед тем как звать преподавателя:

- перезапустите ядро (Kernel → Restart) и выполните сверху вниз подготовку данных и задания, которые будете защищать, — их самопроверки должны пройти;
- проверьте, что во всех полях «Ваш ответ» защищаемых заданий записаны ответы;
- убедитесь, что можете объяснить каждую строку своего кода и показать, где в нём положительный класс, порог и формула каждой метрики.